# Práctica: Clasificación — conceptos básicos

## Introducción

### Objetivo:

Este notebook guía al estudiante en el estudio de los conceptos fundamentales de la clasificación supervisada.
El documento está diseñado para funcionar con distintos tipos de datasets, por lo que incluye bloques opcionales
que pueden activarse o desactivarse según el caso de estudio.

A lo largo del desarrollo se trabajará con una combinación de teoría y práctica. Por una parte, se explican los
conceptos básicos del proceso de clasificación: conjunto de entrenamiento, conjunto de prueba, atributos, etiqueta
de clase, entrenamiento del modelo, predicción y evaluación. Por otra, se implementarán ejemplos con bibliotecas
de Python, además de algunos cálculos básicos de forma explícita para comprender mejor la lógica interna de ciertos
algoritmos.

## Metas:

Al finalizar esta práctica, el estudiante será capaz de:

- Explicar qué es un problema de clasificación y distinguirlo de un problema de regresión.
- Identificar la variable objetivo o etiqueta de clase dentro de un dataset.
- Preparar un conjunto de datos para un problema de clasificación.
- Separar datos en entrenamiento y prueba.
- Construir clasificadores básicos con bibliotecas de Python.
- Interpretar métricas de evaluación como exactitud, matriz de confusión, precisión, recall y F1-score.
- Comprender de manera básica conceptos como entropía, ganancia de información e independencia condicional.
- Adaptar el flujo de trabajo a datasets numéricos, categóricos o mixtos.

## Recomendaciones:

1. Lea primero el contenido en celdas Markdown.
2. Ejecute las celdas de código en orden.
3. Active únicamente los bloques opcionales que correspondan al tipo de dataset utilizado.
4. Antes de entrenar cualquier modelo, verifique siempre:
   - cuál es la columna objetivo,
   - si existen valores faltantes,
   - si hay variables categóricas,
   - si las clases están balanceadas o desbalanceadas.

## Nota importante

Este material está pensado para que el mismo cuaderno pueda usarse con diferentes datasets. Por ello,
en varias secciones aparecen instrucciones como:

```python
# ACTIVAR SOLO SI ...
```

El estudiante deberá decidir qué bloques conservar activos según las características reales de su archivo de datos.


## 1. Fundamento teórico

La clasificación es una forma de análisis de datos que construye modelos capaces de predecir etiquetas categóricas
o clases. A diferencia de la regresión, donde se predicen valores numéricos continuos, en clasificación se predicen
categorías discretas como por ejemplo: `aprobado/reprobado`, `sí/no`, `benigno/maligno` o `seguro/riesgoso`. Este
enfoque forma parte del aprendizaje supervisado porque el modelo aprende a partir de ejemplos donde la clase correcta
ya es conocida. 

De manera general, el proceso de clasificación se desarrolla en dos etapas. Primero se entrena un modelo con un
conjunto de datos etiquetado. Después se evalúa el modelo con datos que no se usaron durante el entrenamiento para
estimar su capacidad de generalización. El texto base del capítulo remarca que medir el desempeño sobre los mismos
datos de entrenamiento produce una estimación optimista, por lo que es necesario usar un conjunto de prueba
independiente. 

El capítulo también presenta a los árboles de decisión como una estructura tipo diagrama de flujo donde cada nodo
interno representa una prueba sobre un atributo, cada rama representa un resultado posible de esa prueba y cada hoja
contiene la clase predicha. Asimismo, se introduce la clasificación bayesiana, en particular el clasificador Naive
Bayes, que supone independencia condicional entre atributos para simplificar el cálculo de probabilidades.


## 2. Flujo general de una práctica de clasificación

En términos prácticos, un flujo de trabajo típico de clasificación sigue estas etapas:

1. Cargar el dataset.
2. Inspecionar su estructura.
3. Identificar la variable objetivo.
4. Limpiar o transformar los datos según sea necesario.
5. Dividir los datos en entrenamiento y prueba.
6. Entrenar uno o varios clasificadores.
7. Generar predicciones.
8. Evaluar el desempeño del modelo.
9. Comparar resultados e interpretar el comportamiento del clasificador.

En este cuaderno se seguirá ese flujo, procurando que cada paso sea lo más reutilizable posible para distintos
problemas.


In [ ]:

# Bibliotecas base para la práctica

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path


## 3. Carga del dataset

En esta práctica no se fija un único dataset. En su lugar, se deja un bloque flexible para que el estudiante lo
adapte. Puede trabajar con un archivo CSV, Excel, o incluso con un dataset de ejemplo de `scikit-learn`.

Antes de continuar, decida cuál será la fuente de datos que utilizará.


In [ ]:

# ============================================================
# CONFIGURACIÓN GENERAL DEL DATASET
# Modifique solo lo necesario para su caso
# ============================================================

RUTA_DATASET = "tu_dataset.csv"   # Cambie este nombre si trabajará con un archivo real
COLUMNA_OBJETIVO = "target"       # Cambie este nombre por la columna de clase real

# Si desea cargar desde archivo:
# df = pd.read_csv(RUTA_DATASET)

# Si desea cargar desde Excel:
# df = pd.read_excel("tu_dataset.xlsx")

# Si desea usar un dataset de ejemplo desde scikit-learn:
# from sklearn.datasets import load_iris
# iris = load_iris(as_frame=True)
# df = iris.frame.copy()
# COLUMNA_OBJETIVO = "target"

# Si su dataset usa otro separador:
# df = pd.read_csv(RUTA_DATASET, sep=";")

print("Revise y active una de las opciones anteriores antes de continuar.")


### Actividad 1

Antes de ejecutar la siguiente parte, responda en una celda Markdown:

1. ¿Cuál es el nombre del dataset que utilizará?
2. ¿Cuál es la columna objetivo?
3. ¿Qué representa cada fila del dataset?
4. ¿Qué representa la clase que se desea predecir?


## 4. Inspección inicial del dataset

Una vez cargado el dataset, conviene revisar:
- el número de filas y columnas,
- los tipos de datos,
- la presencia de valores faltantes,
- algunos registros iniciales.

Este paso permite detectar problemas antes de construir el modelo.


In [ ]:

# ACTIVAR SOLO DESPUÉS DE CARGAR EL DATASET EN LA VARIABLE df

# df.head()
# df.shape
# df.info()


In [ ]:

# Resumen estadístico básico
# ACTIVAR CUANDO EL DATASET YA ESTÉ CARGADO

# df.describe(include="all").T


In [ ]:

# Conteo de valores faltantes por columna
# ACTIVAR CUANDO EL DATASET YA ESTÉ CARGADO

# df.isnull().sum().sort_values(ascending=False)


### Actividad 2

A partir de la inspección inicial, describa lo siguiente:

1. ¿Cuántas filas y columnas tiene su dataset?
2. ¿Qué tipos de datos aparecen?
3. ¿Existen valores faltantes?
4. ¿Hay columnas que parezcan identificadores y no variables predictoras?
5. ¿La columna objetivo parece tener dos clases o varias clases?


## 5. Identificación de la variable objetivo y análisis de clases

En clasificación, la columna objetivo contiene las etiquetas que se desean predecir. Puede estar representada con
texto, números enteros o códigos categóricos.

También es importante conocer la distribución de clases. Si una clase aparece muchísimo más que las demás, el
problema puede estar desbalanceado. El capítulo del libro señala la importancia de evaluar cuidadosamente la
exactitud y otros indicadores, especialmente cuando el interés principal recae en una clase poco frecuente. 


In [ ]:

# ACTIVAR CUANDO df Y COLUMNA_OBJETIVO YA ESTÉN DEFINIDOS

# print(df[COLUMNA_OBJETIVO].value_counts(dropna=False))
# print(df[COLUMNA_OBJETIVO].value_counts(normalize=True, dropna=False))


In [ ]:

# Visualización simple de la distribución de clases
# ACTIVAR CUANDO df Y COLUMNA_OBJETIVO YA ESTÉN DEFINIDOS

# conteo_clases = df[COLUMNA_OBJETIVO].value_counts()
# conteo_clases.plot(kind="bar")
# plt.title("Distribución de clases")
# plt.xlabel("Clase")
# plt.ylabel("Frecuencia")
# plt.show()


### Actividad 3

Con base en el conteo de clases:

1. ¿Las clases están balanceadas o desbalanceadas?
2. ¿Qué problemas podría causar un fuerte desbalance?
3. ¿La exactitud por sí sola sería suficiente para evaluar el modelo en este caso?


## 6. Limpieza y preparación del dataset

No todos los datasets requieren la misma preparación. Por ello, a continuación se presentan bloques opcionales.

### 6.1 Eliminación de columnas no útiles

En muchos casos existen columnas que solo funcionan como identificadores, por ejemplo:
- `id`
- `folio`
- `nombre`
- `matricula`
- `timestamp` muy específico

Estas columnas pueden no aportar poder predictivo real y, en ocasiones, deben eliminarse.


In [ ]:

# ACTIVAR SOLO SI EXISTEN COLUMNAS QUE DEBAN ELIMINARSE

# columnas_a_eliminar = ["id", "nombre"]
# df = df.drop(columns=columnas_a_eliminar, errors="ignore")
# df.head()


### 6.2 Tratamiento de valores faltantes

Los valores faltantes pueden resolverse de varias formas:
- eliminar filas,
- eliminar columnas,
- imputar con media, mediana o moda,
- usar técnicas más avanzadas.

En esta práctica se incluyen opciones sencillas, ustedes pueden agregar otras/especializadas.


In [ ]:

# ACTIVAR SOLO SI DESEA ELIMINAR FILAS CON NULOS
# df = df.dropna()


In [ ]:

# ACTIVAR SOLO SI DESEA IMPUTAR VARIABLES NUMÉRICAS CON LA MEDIA

# columnas_numericas = df.select_dtypes(include=[np.number]).columns.tolist()
# for col in columnas_numericas:
#     if df[col].isnull().sum() > 0:
#         df[col] = df[col].fillna(df[col].mean())


In [ ]:

# ACTIVAR SOLO SI DESEA IMPUTAR VARIABLES CATEGÓRICAS CON LA MODA

# columnas_categoricas = df.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
# for col in columnas_categoricas:
#     if df[col].isnull().sum() > 0:
#         df[col] = df[col].fillna(df[col].mode()[0])


### 6.3 Codificación de variables categóricas

Muchos algoritmos trabajan directamente con números. Si el dataset tiene columnas de texto, se puede aplicar
codificación. Una estrategia común es `one-hot encoding`, que convierte categorías en columnas binarias.

No siempre debe aplicarse a la variable objetivo. Si la clase ya está bien definida como texto y el clasificador la
acepta, puede conservarse. Lo más habitual es codificar solo los atributos predictivos.


In [ ]:

# ACTIVAR SOLO SI EXISTEN VARIABLES CATEGÓRICAS EN LOS PREDICTORES

# X_temporal = df.drop(columns=[COLUMNA_OBJETIVO])
# y_temporal = df[COLUMNA_OBJETIVO]

# X_temporal = pd.get_dummies(X_temporal, drop_first=False)

# df_preparado = X_temporal.copy()
# df_preparado[COLUMNA_OBJETIVO] = y_temporal

# df = df_preparado
# df.head()


### 6.4 Escalamiento de variables

No todos los modelos requieren escalamiento. Por ejemplo, los árboles de decisión no dependen fuertemente de la
escala. En cambio, otros métodos sí pueden beneficiarse de una normalización o estandarización.

En esta práctica principal no se volverá obligatorio, pero se deja un bloque opcional para casos donde el estudiante
quiera experimentar.


In [ ]:

# ACTIVAR SOLO SI DESEA ESCALAR VARIABLES NUMÉRICAS
# Recomendado cuando se comparen modelos sensibles a la escala

# from sklearn.preprocessing import StandardScaler
#
# X_temporal = df.drop(columns=[COLUMNA_OBJETIVO])
# y_temporal = df[COLUMNA_OBJETIVO]
#
# columnas_numericas = X_temporal.select_dtypes(include=[np.number]).columns.tolist()
#
# scaler = StandardScaler()
# X_temporal[columnas_numericas] = scaler.fit_transform(X_temporal[columnas_numericas])
#
# df_escalado = X_temporal.copy()
# df_escalado[COLUMNA_OBJETIVO] = y_temporal
# df = df_escalado


### Actividad 4

Explique qué transformaciones sí fueron necesarias en su dataset y cuáles no.

- ¿Eliminó columnas?
- ¿Imputó valores faltantes?
- ¿Codificó variables categóricas?
- ¿Aplicó escalamiento?
- ¿Por qué tomó esas decisiones?


## 7. Separación entre atributos predictivos y variable objetivo

En la notación del capítulo, una tupla puede representarse como un vector de atributos \(X = (x_1, x_2, ..., x_n)\),
mientras que la etiqueta de clase se denota por \(y\). El objetivo de la clasificación es aprender una función
\(y = f(X)\) que permita asignar una clase a ejemplos no vistos. 


In [ ]:

# ACTIVAR CUANDO df ESTÉ PREPARADO Y LA COLUMNA OBJETIVO SEA CORRECTA

# X = df.drop(columns=[COLUMNA_OBJETIVO])
# y = df[COLUMNA_OBJETIVO]

# print("Dimensión de X:", X.shape)
# print("Dimensión de y:", y.shape)
# X.head()


## 8. División en entrenamiento y prueba

El conjunto de entrenamiento se utiliza para construir el modelo y el conjunto de prueba para estimar su capacidad
de generalización. El capítulo enfatiza que evaluar el clasificador con los mismos datos usados para entrenarlo lleva
a una estimación optimista del desempeño, por lo que debe usarse un conjunto de prueba independiente.


In [ ]:

from sklearn.model_selection import train_test_split


In [ ]:

# ACTIVAR CUANDO X E y YA EXISTAN

# X_train, X_test, y_train, y_test = train_test_split(
#     X,
#     y,
#     test_size=0.2,
#     random_state=42,
#     stratify=y  # Recomendado si desea conservar la proporción de clases
# )

# print("Entrenamiento:", X_train.shape, y_train.shape)
# print("Prueba:", X_test.shape, y_test.shape)


### Actividad 5

Explique con sus palabras:

1. ¿Por qué no conviene evaluar un modelo con el mismo conjunto usado para entrenarlo?
2. ¿Qué significa que un modelo sobreajuste los datos?
3. ¿Por qué puede ser útil usar `stratify=y`?


## 9. Primer modelo: Árbol de decisión

Los árboles de decisión son una de las técnicas más intuitivas de clasificación. Un nodo interno representa una
prueba sobre un atributo, una rama representa un posible resultado de la prueba y una hoja representa la clase final.
El capítulo los presenta como una técnica eficiente y comprensible para explorar datos y construir clasificadores.

Además, el texto explica que durante la construcción del árbol se busca un criterio de partición que separe los datos
de la forma más pura posible, es decir, que deje grupos donde predomine una sola clase. Para ello se utilizan medidas
como la ganancia de información, la razón de ganancia o el índice Gini. 


In [ ]:

from sklearn.tree import DecisionTreeClassifier
from sklearn.tree import plot_tree


In [ ]:

# Entrenamiento básico de un árbol de decisión
# ACTIVAR CUANDO X_train, X_test, y_train, y_test EXISTAN

# modelo_arbol = DecisionTreeClassifier(random_state=42)
# modelo_arbol.fit(X_train, y_train)

# y_pred_arbol = modelo_arbol.predict(X_test)
# y_pred_arbol[:10]


In [ ]:

# Visualización opcional del árbol
# ACTIVAR SOLO SI EL NÚMERO DE VARIABLES Y LA PROFUNDIDAD DEL ÁRBOL NO LO HACEN ILEGIBLE

# plt.figure(figsize=(18, 8))
# plot_tree(modelo_arbol, filled=True, feature_names=X.columns, class_names=True, fontsize=8)
# plt.show()


## 10. Evaluación del modelo

La exactitud o `accuracy` representa el porcentaje de casos correctamente clasificados. El capítulo usa esta idea como
métrica básica al comparar la predicción del clasificador con las etiquetas reales del conjunto de prueba. Sin embargo,
en problemas desbalanceados conviene complementar con otras métricas. 


In [ ]:

from sklearn.metrics import accuracy_score, confusion_matrix, classification_report


In [ ]:

# ACTIVAR CUANDO EXISTA y_pred_arbol

# print("Accuracy del árbol:", accuracy_score(y_test, y_pred_arbol))
# print("\nMatriz de confusión:")
# print(confusion_matrix(y_test, y_pred_arbol))
# print("\nReporte de clasificación:")
# print(classification_report(y_test, y_pred_arbol))


### Interpretación sugerida

- **Accuracy**: proporción total de aciertos.
- **Matriz de confusión**: muestra cuántos ejemplos de cada clase fueron clasificados correcta o incorrectamente.
- **Precision**: de los casos predichos como positivos, cuántos realmente lo eran.
- **Recall**: de los casos positivos reales, cuántos fueron detectados.
- **F1-score**: balance entre precision y recall.

Estas métricas deben interpretarse a la luz del problema real. No significa lo mismo equivocarse en un problema de
recomendación de compras que en un problema de diagnóstico médico.


### Actividad 6

Interprete los resultados de su árbol de decisión:

1. ¿Qué accuracy obtuvo?
2. ¿Qué clase fue la mejor clasificada?
3. ¿Qué clase presentó más errores?
4. ¿La matriz de confusión sugiere un patrón de error importante?


## 11. Comprensión conceptual: entropía y ganancia de información

El capítulo introduce la entropía como una medida de impureza de un conjunto de datos. Si todas las observaciones
pertenecen a la misma clase, la impureza es baja. Si las clases están muy mezcladas, la impureza es mayor.

La entropía de un conjunto \(D\) puede expresarse como:

\[
Info(D) = -\sum_{i=1}^{m} p_i \log_2(p_i)
\]

donde \(p_i\) es la proporción de elementos de la clase \(i\). La ganancia de información compara la impureza
antes y después de dividir el conjunto usando un atributo. El atributo que más reduce la impureza es un buen
candidato para construir el árbol. 


In [ ]:

def entropia_etiquetas(y):
    valores, conteos = np.unique(y, return_counts=True)
    probabilidades = conteos / conteos.sum()
    return -np.sum(probabilidades * np.log2(probabilidades + 1e-12))


In [ ]:

# Ejemplo simple de uso de entropía
ejemplo_clases = np.array(["A", "A", "A", "B", "B", "C"])
print("Entropía del ejemplo:", entropia_etiquetas(ejemplo_clases))


In [ ]:

# ACTIVAR SI DESEA CALCULAR LA ENTROPÍA DE SU VARIABLE OBJETIVO

# print("Entropía de la columna objetivo:", entropia_etiquetas(df[COLUMNA_OBJETIVO]))


### Implementación didáctica de ganancia de información para atributos categóricos

El siguiente bloque no sustituye una biblioteca profesional, pero ayuda a comprender cómo un atributo puede reducir
la impureza de un conjunto.


In [ ]:

def ganancia_informacion_categorica(df_local, atributo, objetivo):
    entropia_total = entropia_etiquetas(df_local[objetivo])
    valores = df_local[atributo].dropna().unique()

    entropia_condicional = 0
    n_total = len(df_local)

    for valor in valores:
        subconjunto = df_local[df_local[atributo] == valor][objetivo]
        peso = len(subconjunto) / n_total
        entropia_condicional += peso * entropia_etiquetas(subconjunto)

    return entropia_total - entropia_condicional


In [ ]:

# ACTIVAR SOLO SI DESEA PROBAR GANANCIA DE INFORMACIÓN EN ATRIBUTOS CATEGÓRICOS
# Y SI SU DATASET AÚN CONSERVA ALGUNAS COLUMNAS CATEGÓRICAS SIN DUMMIES

# for col in df.columns:
#     if col != COLUMNA_OBJETIVO and df[col].dtype == "object":
#         ganancia = ganancia_informacion_categorica(df, col, COLUMNA_OBJETIVO)
#         print(f"{col}: {ganancia:.6f}")


### Actividad 7

Responda:

1. ¿Qué significa que un conjunto sea puro?
2. ¿Qué indica una entropía alta?
3. ¿Por qué un atributo con alta ganancia de información puede ser útil para dividir el dataset?
4. ¿La ganancia de información decide directamente la clase final o solo ayuda a seleccionar divisiones?


## 12. Segundo modelo: Naive Bayes

El capítulo describe a los clasificadores bayesianos como clasificadores estadísticos que pueden estimar la
probabilidad de pertenencia a cada clase. En particular, el modelo Naive Bayes asume independencia condicional entre
atributos respecto de la clase, lo cual simplifica mucho el cálculo. Aunque esta suposición es fuerte y a veces no
se cumple exactamente, el método suele ofrecer muy buen desempeño en muchos problemas reales.


In [ ]:

from sklearn.naive_bayes import GaussianNB


### Nota práctica sobre Naive Bayes

- `GaussianNB` es adecuado cuando los predictores son numéricos continuos o se aproximan a una distribución gaussiana.
- Para texto o conteos discretos pueden existir otras variantes como `MultinomialNB` o `BernoulliNB`.
- Si su dataset quedó completamente numérico después del preprocesamiento, `GaussianNB` puede ser una primera prueba.


In [ ]:

# Entrenamiento de Naive Bayes
# ACTIVAR CUANDO X_train, X_test, y_train, y_test EXISTAN

# modelo_nb = GaussianNB()
# modelo_nb.fit(X_train, y_train)

# y_pred_nb = modelo_nb.predict(X_test)
# y_pred_nb[:10]


In [ ]:

# Evaluación de Naive Bayes
# ACTIVAR CUANDO EXISTA y_pred_nb

# print("Accuracy de Naive Bayes:", accuracy_score(y_test, y_pred_nb))
# print("\nMatriz de confusión:")
# print(confusion_matrix(y_test, y_pred_nb))
# print("\nReporte de clasificación:")
# print(classification_report(y_test, y_pred_nb))


### Actividad 8

Compare el árbol de decisión con Naive Bayes:

1. ¿Cuál obtuvo mejor accuracy?
2. ¿Cuál parece comportarse mejor para cada clase?
3. ¿Cuál considera más interpretable?
4. ¿Qué ventajas y desventajas observó en cada uno?


## 13. Comparación directa de modelos

En una práctica de ciencia de datos no basta con entrenar un solo modelo. Es importante comparar alternativas y
entender que distintos clasificadores pueden responder de forma diferente según la naturaleza del dataset.


In [ ]:

# Comparación simple de accuracies
# ACTIVAR SOLO SI YA ENTRENÓ AMBOS MODELOS

# resultados = pd.DataFrame({
#     "Modelo": ["Árbol de decisión", "Naive Bayes"],
#     "Accuracy": [
#         accuracy_score(y_test, y_pred_arbol),
#         accuracy_score(y_test, y_pred_nb)
#     ]
# })
#
# resultados


In [ ]:

# Gráfica comparativa
# ACTIVAR SOLO SI YA GENERÓ EL DATAFRAME resultados

# resultados.plot(x="Modelo", y="Accuracy", kind="bar", legend=False)
# plt.ylim(0, 1)
# plt.ylabel("Accuracy")
# plt.title("Comparación de modelos de clasificación")
# plt.show()


## 14. Ajuste básico del árbol de decisión

Un árbol de decisión muy profundo puede sobreajustar. El capítulo menciona la poda como una estrategia para reducir
ramas poco confiables y mejorar la generalización. En la práctica moderna, una forma sencilla de aproximarse a esta
idea consiste en limitar la profundidad del árbol o imponer restricciones mínimas al crecimiento.


In [ ]:

# ACTIVAR SI DESEA PROBAR UN ÁRBOL MÁS CONTROLADO

# modelo_arbol_controlado = DecisionTreeClassifier(
#     max_depth=3,
#     min_samples_split=4,
#     min_samples_leaf=2,
#     random_state=42
# )
#
# modelo_arbol_controlado.fit(X_train, y_train)
# y_pred_arbol_controlado = modelo_arbol_controlado.predict(X_test)
#
# print("Accuracy del árbol controlado:", accuracy_score(y_test, y_pred_arbol_controlado))
# print("\nMatriz de confusión:")
# print(confusion_matrix(y_test, y_pred_arbol_controlado))
# print("\nReporte de clasificación:")
# print(classification_report(y_test, y_pred_arbol_controlado))


### Actividad 9

Reflexione:

1. ¿Al limitar la profundidad del árbol mejoró o empeoró el desempeño?
2. ¿Qué relación observa entre complejidad del modelo e interpretación?
3. ¿Un modelo más complejo siempre clasifica mejor datos nuevos?


## 15. Validación cruzada básica

Además de una sola división entrenamiento-prueba, puede ser útil utilizar validación cruzada para obtener una
estimación más estable del desempeño. Esta parte no reemplaza el flujo principal de la práctica, pero se incluye como
extensión útil.


In [ ]:

from sklearn.model_selection import cross_val_score


In [ ]:

# ACTIVAR SOLO SI DESEA ESTIMAR DESEMPEÑO MEDIANTE VALIDACIÓN CRUZADA

# modelo_cv = DecisionTreeClassifier(random_state=42)
# puntajes = cross_val_score(modelo_cv, X, y, cv=5, scoring="accuracy")
#
# print("Puntajes por partición:", puntajes)
# print("Accuracy promedio:", puntajes.mean())
# print("Desviación estándar:", puntajes.std())


### Actividad 10

Explique la diferencia entre:

- una sola división entrenamiento-prueba,
- validación cruzada.

¿En qué casos podría ser más conveniente la validación cruzada?


## 16. Recomendaciones según el tipo de dataset

### Caso A: dataset completamente numérico
- Puede trabajar con `GaussianNB` y árboles sin demasiados cambios.
- Revise si existen escalas muy distintas y decida si vale la pena estandarizar.

### Caso B: dataset mixto (numérico + categórico)
- Elimine identificadores no útiles.
- Codifique los predictores categóricos.
- Revise si la variable objetivo necesita mantenerse como texto o convertirse a códigos.

### Caso C: dataset con muchas categorías de texto
- Evalúe cuidadosamente la codificación.
- Piense si el problema requiere un tratamiento especializado.
- No convierta todo automáticamente sin interpretar qué representa cada columna.

### Caso D: dataset desbalanceado
- No se conforme solo con accuracy.
- Observe recall, precision y F1-score por clase.
- Analice si la clase minoritaria es precisamente la más importante.


## 17. Ejercicio integrador

Utilice este mismo cuaderno con un dataset distinto al utilizado originalmente y realice lo siguiente:

1. Ajuste la carga de datos.
2. Identifique la nueva columna objetivo.
3. Prepare el dataset.
4. Entrene un árbol de decisión.
5. Entrene un modelo Naive Bayes.
6. Compare ambos modelos.
7. Redacte una conclusión de media cuartilla donde explique:
   - cuál modelo funcionó mejor,
   - qué dificultades encontró en el preprocesamiento,
   - qué métrica considera más útil para su problema,
   - qué mejoraría si continuara el proyecto.


## 18. Preguntas de cierre

1. ¿Qué diferencia existe entre clasificación y regresión?
2. ¿Qué significa que la clasificación sea un proceso supervisado?
3. ¿Por qué se separan datos de entrenamiento y de prueba?
4. ¿Qué representa la variable objetivo?
5. ¿Qué aporta la matriz de confusión que no aporta directamente el accuracy?
6. ¿Qué significa entropía en el contexto de árboles de decisión?
7. ¿Qué hace Naive Bayes y qué suposición simplificadora utiliza?
8. ¿Por qué no existe un único clasificador mejor para todos los datasets?


## 19. Conclusión general

La clasificación constituye una de las tareas centrales de la ciencia de datos. Su valor no radica únicamente en
obtener una predicción, sino en comprender cómo los atributos de un conjunto de datos se relacionan con una clase
de interés. En esta práctica se recorrió el flujo básico de trabajo para construir clasificadores, desde la carga y
preparación del dataset hasta la evaluación e interpretación de resultados.

El árbol de decisión permitió estudiar la lógica de partición del espacio de datos y relacionarla con conceptos como
entropía, pureza y ganancia de información. Por su parte, Naive Bayes permitió introducir una perspectiva
probabilística de la clasificación. En ambos casos, el objetivo no fue solo ejecutar bibliotecas, sino comprender qué
hace cada técnica y en qué condiciones puede ser útil.

A partir de este punto, el estudiante queda preparado para avanzar hacia temas posteriores como selección de
atributos, mejora de clasificadores, manejo de desbalance, comparación más rigurosa de modelos y clasificación con
datasets de mayor tamaño.


## 20. Sugerencia para entrega

Se recomienda entregar:

1. El cuaderno ejecutado.
2. El dataset utilizado o una referencia clara a su origen.
3. Una breve interpretación de resultados (pueden hacer una subsección para ello).
4. Respuestas a las actividades incluidas en el documento.
5. Una conclusión personal sobre el comportamiento de los modelos probados.
